# Short-Term Video Event Prediction from RGB Features

Compact experiments for egocentric next-action anticipation with pre-extracted
RGB feature sequences. The target is feature-based modeling under Kaggle
constraints, not raw video training or large visual backbone training.

Input: `X = (f_1, ..., f_T)`, where each `f_t` is an RGB feature vector.

Output: next action as verb label `v`, noun label `n`, and joint pair `(v, n)`.



In [38]:
from __future__ import annotations

import csv
import json
import math
import os
import pickle
import random
import shutil
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, Subset

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None




## Environment and Hardware Check



In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def get_output_dir() -> Path:
    override = os.environ.get("OUTPUT_DIR")
    if override:
        return Path(override)
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists():
        return kaggle_working
    return Path.cwd() / "kaggle_working"


OUTPUT_DIR = get_output_dir()
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = "uninitialized_run"
RUN_DIR = OUTPUT_DIR
RESULTS_PATH = OUTPUT_DIR / "results_rgb_features.csv"
PLOT_PATH = OUTPUT_DIR / "quality_efficiency.png"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python cwd: {Path.cwd()}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Using device 0: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Device memory: {props.total_memory / 1024**3:.2f} GiB")
else:
    print("Running on CPU. This is fine for smoke tests; Kaggle T4 is preferred.")




## Config



In [ ]:
def env_flag(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


def env_optional_str(name: str, default: Optional[str] = None) -> Optional[str]:
    value = os.environ.get(name)
    if value is None:
        return default
    value = value.strip()
    return value or default


@dataclass
class Config:
    seed: int = 42
    max_len: int = 32
    batch_size: int = 64
    epochs: int = 5
    fast_dev_run: bool = env_flag("FAST_DEV_RUN", False)
    run_tiny_rmt: bool = env_flag("RUN_TINY_RMT", False)
    sweep_seeds: Optional[Tuple[int, ...]] = None
    models_to_run: Optional[Tuple[str, ...]] = None  # e.g. ("gru",), ("attn_gru",), ("meanpool_mlp", "gru")
    max_train_samples: Optional[int] = None
    max_val_samples: Optional[int] = None
    lr: float = 2e-4
    weight_decay: float = 1e-2
    hidden_dim: int = 256
    dropout: float = 0.2
    grad_clip_norm: float = 1.0
    num_workers: int = 0
    early_stop_patience: int = 3
    early_stop_min_epochs: int = 0
    lr_scheduler: str = "none"  # "none" or "cosine"
    min_lr_ratio: float = 0.05
    action_loss_weight: float = 0.0
    action_score_weight: float = 0.0
    action_score_weights: Optional[Tuple[float, ...]] = None
    prior_alpha: float = 1.0
    prior_lambdas: Tuple[float, ...] = (0.0, 0.1, 0.3, 0.5, 1.0)
    truncate_mode: str = "uniform"  # "uniform" or "last"
    run_name: Optional[str] = env_optional_str("RUN_NAME", None)
    save_latest_copy: bool = True
    synthetic_train_samples: int = 512
    synthetic_val_samples: int = 128
    synthetic_feature_dim: int = 512
    synthetic_num_verbs: int = 12
    synthetic_num_nouns: int = 24


CFG = Config()

if CFG.fast_dev_run:
    CFG.epochs = 1
    CFG.batch_size = min(CFG.batch_size, 32)
    CFG.max_train_samples = 96
    CFG.max_val_samples = 48
    CFG.synthetic_train_samples = 96
    CFG.synthetic_val_samples = 48
    print("FAST_DEV_RUN is enabled: using tiny subsets and one epoch.")

set_seed(CFG.seed)
print(CFG)




## Data Discovery

This notebook searches for common feature and metadata formats in Kaggle input,
Kaggle working, and the current project directory. It never downloads raw video
or full external datasets.



In [ ]:
CANDIDATE_EXTENSIONS = (".npy", ".npz", ".pt", ".pth", ".pkl", ".csv", ".json", ".mdb", ".lmdb")


def discover_data_files(
    roots: Optional[Sequence[Path]] = None,
    extensions: Sequence[str] = CANDIDATE_EXTENSIONS,
    max_files: int = 10_000,
) -> Dict[str, List[Path]]:
    if roots is None:
        roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]

    found: Dict[str, List[Path]] = {ext: [] for ext in extensions}
    seen: set[Path] = set()

    for root in roots:
        if not root.exists():
            print(f"[discover] missing root: {root}")
            continue
        print(f"[discover] scanning: {root}")
        for ext in extensions:
            try:
                for path in root.rglob(f"*{ext}"):
                    if path in seen:
                        continue
                    seen.add(path)
                    found[ext].append(path)
                    if sum(len(v) for v in found.values()) >= max_files:
                        print(f"[discover] reached max_files={max_files}; stopping early")
                        return found
            except OSError as exc:
                print(f"[discover] skipped {root} pattern *{ext}: {exc}")

    total = sum(len(v) for v in found.values())
    print(f"[discover] total candidate files/directories: {total}")
    for ext, paths in found.items():
        if paths:
            print(f"  {ext}: {len(paths)}")
            for example in paths[:5]:
                print(f"    - {example}")
    return found


DISCOVERED_FILES = discover_data_files()




## Dataset and Collate Function



In [ ]:
FEATURE_KEYS = ("features", "feature", "rgb_features", "rgb", "x", "X", "feat", "feats")
VERB_KEYS = ("verb", "verbs", "verb_label", "verb_labels", "verb_class", "verb_id", "verb_ids")
NOUN_KEYS = ("noun", "nouns", "noun_label", "noun_labels", "noun_class", "noun_id", "noun_ids")
SPLIT_KEYS = ("split", "splits", "subset", "partition")
FEATURE_PATH_KEYS = ("feature_path", "features_path", "rgb_feature_path", "npy_path", "path", "filepath", "file", "filename")


def find_key(mapping: Mapping[str, Any], candidates: Sequence[str]) -> Optional[str]:
    lower_to_key = {str(k).lower(): k for k in mapping.keys()}
    for cand in candidates:
        key = lower_to_key.get(cand.lower())
        if key is not None:
            return key
    for key in mapping.keys():
        key_lower = str(key).lower()
        if any(cand.lower() in key_lower for cand in candidates):
            return str(key)
    return None


def to_numpy(value: Any) -> np.ndarray:
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def safe_torch_load(path: Path) -> Any:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def sorted_unique(values: Sequence[Any]) -> List[Any]:
    unique = list(dict.fromkeys(values))
    try:
        return sorted(unique, key=lambda x: int(x))
    except (TypeError, ValueError):
        return sorted(unique, key=lambda x: str(x))


def encode_label_values(values: Sequence[Any]) -> Tuple[List[int], Dict[Any, int]]:
    classes = sorted_unique(list(values))
    mapping = {value: idx for idx, value in enumerate(classes)}
    return [mapping[value] for value in values], mapping


def normalize_feature_array(feature: Any, expected_dim: Optional[int] = None) -> torch.Tensor:
    array = to_numpy(feature)
    if array.dtype == object:
        array = np.asarray(array.tolist())
    tensor = torch.as_tensor(array, dtype=torch.float32)
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)
    elif tensor.ndim > 2:
        tensor = tensor.reshape(tensor.shape[0], -1)
    if tensor.ndim != 2:
        raise ValueError(f"Feature must have shape [T, D] or [D], got {tuple(tensor.shape)}")
    if tensor.shape[0] < 1 or tensor.shape[1] < 1:
        raise ValueError(f"Feature sequence must be non-empty, got {tuple(tensor.shape)}")
    if expected_dim is not None and tensor.shape[1] != expected_dim:
        raise ValueError(f"Feature dim mismatch: expected {expected_dim}, got {tensor.shape[1]}")
    return tensor


class FeatureSequenceDataset(Dataset):
    def __init__(self, samples: Sequence[Dict[str, Any]], name: str, feature_dim: Optional[int] = None):
        self.samples = list(samples)
        self.name = name
        self.feature_dim = feature_dim
        self.verbs = [int(sample["verb"]) for sample in self.samples]
        self.nouns = [int(sample["noun"]) for sample in self.samples]
        if not self.samples:
            raise ValueError(f"{name} dataset is empty")

    def __len__(self) -> int:
        return len(self.samples)

    def iter_labels(self) -> Iterable[Tuple[int, int]]:
        return zip(self.verbs, self.nouns)

    def _load_feature_from_path(self, path: Path) -> Any:
        suffix = path.suffix.lower()
        if suffix == ".npy":
            return np.load(path, allow_pickle=True)
        if suffix == ".npz":
            data = np.load(path, allow_pickle=True)
            key = find_key({k: None for k in data.files}, FEATURE_KEYS)
            if key is None:
                raise ValueError(f"No feature-like key found in {path}; keys={data.files}")
            return data[key]
        if suffix in {".pt", ".pth"}:
            data = safe_torch_load(path)
            if isinstance(data, Mapping):
                key = find_key(data, FEATURE_KEYS)
                if key is None:
                    raise ValueError(f"No feature-like key found in {path}; keys={list(data.keys())[:20]}")
                return data[key]
            return data
        if suffix == ".pkl":
            with path.open("rb") as f:
                data = pickle.load(f)
            if isinstance(data, Mapping):
                key = find_key(data, FEATURE_KEYS)
                if key is None:
                    raise ValueError(f"No feature-like key found in {path}; keys={list(data.keys())[:20]}")
                return data[key]
            return data
        raise ValueError(f"Unsupported feature path suffix for sample loading: {path}")

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        sample = self.samples[idx]
        if "features" in sample:
            features = sample["features"]
        elif "feature_path" in sample:
            features = self._load_feature_from_path(Path(sample["feature_path"]))
        else:
            raise KeyError("Sample must contain either 'features' or 'feature_path'")
        feature_tensor = normalize_feature_array(features, expected_dim=self.feature_dim)
        verb = torch.tensor(int(sample["verb"]), dtype=torch.long)
        noun = torch.tensor(int(sample["noun"]), dtype=torch.long)
        return feature_tensor, verb, noun


def pad_or_truncate_sequence(features: torch.Tensor, max_len: int, mode: str = "uniform") -> Tuple[torch.Tensor, torch.Tensor]:
    features = normalize_feature_array(features)
    length, dim = features.shape
    if length >= max_len:
        if mode == "uniform" and max_len > 1:
            indices = torch.linspace(0, length - 1, max_len).round().long()
            out = features.index_select(0, indices)
        elif mode == "last":
            out = features[-max_len:]
        else:
            out = features[:max_len]
        mask = torch.ones(max_len, dtype=torch.bool)
        return out, mask

    out = torch.zeros(max_len, dim, dtype=features.dtype)
    out[:length] = features
    mask = torch.zeros(max_len, dtype=torch.bool)
    mask[:length] = True
    return out, mask


def make_collate_fn(max_len: int, truncate_mode: str):
    def collate_fn(batch: Sequence[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        features, verbs, nouns = zip(*batch)
        dims = {int(feature.shape[-1]) for feature in features}
        if len(dims) != 1:
            raise ValueError(f"All feature vectors in a batch must share dim D; got dims={sorted(dims)}")
        padded, masks = zip(*(pad_or_truncate_sequence(feature, max_len, truncate_mode) for feature in features))
        return {
            "features": torch.stack(padded, dim=0),
            "mask": torch.stack(masks, dim=0),
            "verb": torch.stack(verbs, dim=0),
            "noun": torch.stack(nouns, dim=0),
        }

    return collate_fn




## Real Feature Dataset Loader

`load_real_feature_dataset()` is intentionally adapter-style. If your feature
bundle has a different layout, edit the TODO-marked adapter block and keep the
returned interface unchanged:

`train_dataset, val_dataset, num_verbs, num_nouns, feature_dim, dataset_mode`



In [ ]:
def flat_discovered_files(discovered: Dict[str, List[Path]]) -> List[Path]:
    return [path for paths in discovered.values() for path in paths]


def build_basename_index(paths: Sequence[Path]) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = {}
    for path in paths:
        index.setdefault(path.name, []).append(path)
    return index


def resolve_feature_path(raw_value: Any, metadata_path: Path, basename_index: Dict[str, List[Path]]) -> Optional[Path]:
    if raw_value is None or (isinstance(raw_value, float) and math.isnan(raw_value)):
        return None
    raw_text = str(raw_value).strip()
    if not raw_text:
        return None

    candidates = [Path(raw_text), metadata_path.parent / raw_text]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = basename_index.get(Path(raw_text).name, [])
    if len(matches) == 1:
        return matches[0]
    return None


def split_samples(
    samples: Sequence[Dict[str, Any]],
    split_values: Optional[Sequence[Any]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    samples = list(samples)
    if split_values is not None:
        train, val, test_like = [], [], []
        for sample, split in zip(samples, split_values):
            split_lower = str(split).strip().lower()
            if split_lower in {"train", "training"}:
                train.append(sample)
            elif split_lower in {"val", "valid", "validation"}:
                val.append(sample)
            elif split_lower in {"test", "testing"}:
                test_like.append(sample)
        if train and val:
            return train, val
        if train and test_like:
            print("[loader] using test split as validation because no validation split was found")
            return train, test_like

    rng = random.Random(seed)
    indices = list(range(len(samples)))
    rng.shuffle(indices)
    val_size = max(1, int(0.2 * len(indices)))
    val_idx = set(indices[:val_size])
    train = [sample for i, sample in enumerate(samples) if i not in val_idx]
    val = [sample for i, sample in enumerate(samples) if i in val_idx]
    return train, val


def cap_dataset(dataset: Dataset, max_samples: Optional[int], seed: int) -> Dataset:
    if max_samples is None or len(dataset) <= max_samples:
        return dataset
    rng = random.Random(seed)
    indices = list(range(len(dataset)))
    rng.shuffle(indices)
    return Subset(dataset, sorted(indices[:max_samples]))


def make_datasets_from_samples(
    samples: Sequence[Dict[str, Any]],
    split_values: Optional[Sequence[Any]],
    cfg: Config,
    source_name: str,
) -> Optional[Tuple[Dataset, Dataset, int, int, int, str]]:
    if len(samples) < 2:
        return None

    verb_values = [sample["verb"] for sample in samples]
    noun_values = [sample["noun"] for sample in samples]
    encoded_verbs, verb_map = encode_label_values(verb_values)
    encoded_nouns, noun_map = encode_label_values(noun_values)
    encoded_samples = []
    for sample, verb, noun in zip(samples, encoded_verbs, encoded_nouns):
        new_sample = dict(sample)
        new_sample["verb"] = verb
        new_sample["noun"] = noun
        encoded_samples.append(new_sample)

    train_samples, val_samples = split_samples(encoded_samples, split_values, cfg.seed)
    if not train_samples or not val_samples:
        return None

    probe_dataset = FeatureSequenceDataset([train_samples[0]], name=f"{source_name}-probe")
    probe_features, _, _ = probe_dataset[0]
    feature_dim = int(probe_features.shape[-1])

    train_dataset = FeatureSequenceDataset(train_samples, name=f"{source_name}-train", feature_dim=feature_dim)
    val_dataset = FeatureSequenceDataset(val_samples, name=f"{source_name}-val", feature_dim=feature_dim)

    train_dataset = cap_dataset(train_dataset, cfg.max_train_samples, cfg.seed)
    val_dataset = cap_dataset(val_dataset, cfg.max_val_samples, cfg.seed + 1)

    print(f"[loader] using real feature data from {source_name}")
    print(f"[loader] train samples: {len(train_dataset)}, val samples: {len(val_dataset)}")
    print(f"[loader] num_verbs={len(verb_map)}, num_nouns={len(noun_map)}, feature_dim={feature_dim}")
    print(f"[loader] first feature shape before padding: {tuple(probe_features.shape)}")
    return train_dataset, val_dataset, len(verb_map), len(noun_map), feature_dim, "real"


def try_load_csv_metadata(
    csv_paths: Sequence[Path],
    basename_index: Dict[str, List[Path]],
    cfg: Config,
) -> Optional[Tuple[Dataset, Dataset, int, int, int, str]]:
    if pd is None:
        print("[loader] pandas is unavailable; skipping CSV metadata adapters")
        return None

    for csv_path in csv_paths:
        try:
            frame = pd.read_csv(csv_path)
        except Exception as exc:
            print(f"[loader] could not read CSV {csv_path}: {exc}")
            continue
        if frame.empty:
            continue

        columns = {col: None for col in frame.columns}
        feature_col = find_key(columns, FEATURE_PATH_KEYS)
        verb_col = find_key(columns, VERB_KEYS)
        noun_col = find_key(columns, NOUN_KEYS)
        split_col = find_key(columns, SPLIT_KEYS)

        if feature_col is None or verb_col is None or noun_col is None:
            continue

        samples: List[Dict[str, Any]] = []
        split_values: List[Any] = []
        missing_features = 0
        for row in frame.to_dict(orient="records"):
            feature_path = resolve_feature_path(row[feature_col], csv_path, basename_index)
            if feature_path is None:
                missing_features += 1
                continue
            samples.append({"feature_path": feature_path, "verb": row[verb_col], "noun": row[noun_col]})
            if split_col is not None:
                split_values.append(row[split_col])

        if missing_features:
            print(f"[loader] {csv_path.name}: skipped {missing_features} rows with unresolved feature paths")

        split_values_or_none = split_values if split_col is not None else None

        result = make_datasets_from_samples(samples, split_values_or_none, cfg, source_name=csv_path.name)
        if result is not None:
            return result

    return None


def array_samples_from_mapping(data: Mapping[str, Any], source_name: str) -> Optional[Tuple[List[Dict[str, Any]], Optional[List[Any]]]]:
    feature_key = find_key(data, FEATURE_KEYS)
    verb_key = find_key(data, VERB_KEYS)
    noun_key = find_key(data, NOUN_KEYS)
    split_key = find_key(data, SPLIT_KEYS)
    if feature_key is None or verb_key is None or noun_key is None:
        return None

    features = data[feature_key]
    verbs = list(to_numpy(data[verb_key]).reshape(-1))
    nouns = list(to_numpy(data[noun_key]).reshape(-1))
    if len(verbs) != len(nouns):
        raise ValueError(f"{source_name}: verb/noun label lengths differ: {len(verbs)} vs {len(nouns)}")

    num_samples = len(verbs)
    feature_array = features
    if isinstance(feature_array, (list, tuple)):
        first_dim = len(feature_array)
    elif isinstance(feature_array, torch.Tensor):
        first_dim = int(feature_array.shape[0]) if feature_array.ndim > 0 else 1
    else:
        try:
            feature_array = np.asarray(feature_array)
        except ValueError:
            feature_array = np.asarray(feature_array, dtype=object)
        first_dim = int(feature_array.shape[0]) if feature_array.ndim > 0 else 1

    if first_dim != num_samples:
        if num_samples == 1:
            feature_items = [features]
        else:
            raise ValueError(f"{source_name}: features first dim {first_dim} does not match labels {num_samples}")
    else:
        feature_items = [features[i] for i in range(num_samples)]

    samples = [
        {"features": feature_items[i], "verb": verbs[i], "noun": nouns[i]}
        for i in range(num_samples)
    ]
    split_values = list(to_numpy(data[split_key]).reshape(-1)) if split_key is not None else None
    return samples, split_values


def try_load_npz_files(
    npz_paths: Sequence[Path],
    cfg: Config,
) -> Optional[Tuple[Dataset, Dataset, int, int, int, str]]:
    for path in npz_paths:
        try:
            npz = np.load(path, allow_pickle=True)
            data = {key: npz[key] for key in npz.files}
            converted = array_samples_from_mapping(data, path.name)
            if converted is None:
                continue
            samples, split_values = converted
            result = make_datasets_from_samples(samples, split_values, cfg, source_name=path.name)
            if result is not None:
                return result
        except Exception as exc:
            print(f"[loader] could not adapt NPZ {path}: {exc}")
    return None


def try_load_torch_or_pickle_files(
    paths: Sequence[Path],
    cfg: Config,
) -> Optional[Tuple[Dataset, Dataset, int, int, int, str]]:
    for path in paths:
        try:
            if path.suffix.lower() in {".pt", ".pth"}:
                data = safe_torch_load(path)
            else:
                with path.open("rb") as f:
                    data = pickle.load(f)

            if isinstance(data, Mapping) and {"train", "val"}.issubset({str(k).lower() for k in data.keys()}):
                lower_to_key = {str(k).lower(): k for k in data.keys()}
                train_part = data[lower_to_key["train"]]
                val_part = data[lower_to_key["val"]]
                train_converted = array_samples_from_mapping(train_part, f"{path.name}:train") if isinstance(train_part, Mapping) else None
                val_converted = array_samples_from_mapping(val_part, f"{path.name}:val") if isinstance(val_part, Mapping) else None
                if train_converted is not None and val_converted is not None:
                    train_samples, _ = train_converted
                    val_samples, _ = val_converted
                    all_samples = train_samples + val_samples
                    split_values = ["train"] * len(train_samples) + ["val"] * len(val_samples)
                    result = make_datasets_from_samples(all_samples, split_values, cfg, source_name=path.name)
                    if result is not None:
                        return result

            if isinstance(data, Mapping):
                converted = array_samples_from_mapping(data, path.name)
                if converted is None:
                    continue
                samples, split_values = converted
                result = make_datasets_from_samples(samples, split_values, cfg, source_name=path.name)
                if result is not None:
                    return result

            if isinstance(data, (list, tuple)) and data and isinstance(data[0], Mapping):
                samples = []
                split_values = []
                for record in data:
                    feature_key = find_key(record, FEATURE_KEYS)
                    feature_path_key = find_key(record, FEATURE_PATH_KEYS)
                    verb_key = find_key(record, VERB_KEYS)
                    noun_key = find_key(record, NOUN_KEYS)
                    split_key = find_key(record, SPLIT_KEYS)
                    if verb_key is None or noun_key is None or (feature_key is None and feature_path_key is None):
                        samples = []
                        break
                    sample = {"verb": record[verb_key], "noun": record[noun_key]}
                    if feature_key is not None:
                        sample["features"] = record[feature_key]
                    else:
                        sample["feature_path"] = Path(record[feature_path_key])
                    samples.append(sample)
                    split_values.append(record[split_key] if split_key is not None else None)
                if samples:
                    result = make_datasets_from_samples(samples, split_values if any(v is not None for v in split_values) else None, cfg, source_name=path.name)
                    if result is not None:
                        return result
        except Exception as exc:
            print(f"[loader] could not adapt {path}: {exc}")
    return None


def load_real_feature_dataset(
    discovered: Dict[str, List[Path]],
    cfg: Config,
) -> Optional[Tuple[Dataset, Dataset, int, int, int, str]]:
    all_paths = flat_discovered_files(discovered)
    basename_index = build_basename_index(all_paths)

    # TODO: Add a project-specific adapter here if your EPIC-KITCHENS feature
    # files use a custom LMDB/MDB layout or nested metadata schema.
    adapters = [
        ("CSV metadata", lambda: try_load_csv_metadata(discovered.get(".csv", []), basename_index, cfg)),
        ("NPZ arrays", lambda: try_load_npz_files(discovered.get(".npz", []), cfg)),
        ("Torch/Pickle dictionaries", lambda: try_load_torch_or_pickle_files(discovered.get(".pt", []) + discovered.get(".pth", []) + discovered.get(".pkl", []), cfg)),
    ]

    for name, adapter in adapters:
        print(f"[loader] trying adapter: {name}")
        result = adapter()
        if result is not None:
            return result

    if discovered.get(".mdb") or discovered.get(".lmdb"):
        print("[loader] LMDB/MDB candidates were found, but no generic schema is assumed.")
        print("[loader] Add a custom reader in load_real_feature_dataset() for that dataset layout.")

    return None


def make_synthetic_datasets(cfg: Config) -> Tuple[Dataset, Dataset, int, int, int, str]:
    print("=" * 80)
    print("SYNTHETIC SMOKE-TEST MODE")
    print("No usable real RGB feature dataset was found. Results below are not real research results.")
    print("They only verify that training, metrics, priors, result tables, and plots work.")
    print("=" * 80)

    rng = np.random.default_rng(cfg.seed)
    num_verbs = cfg.synthetic_num_verbs
    num_nouns = cfg.synthetic_num_nouns
    feature_dim = cfg.synthetic_feature_dim
    verb_proto = rng.normal(size=(num_verbs, feature_dim)).astype(np.float32)
    noun_proto = rng.normal(size=(num_nouns, feature_dim)).astype(np.float32)
    action_bias = rng.normal(scale=0.35, size=(num_verbs, num_nouns, feature_dim)).astype(np.float32)

    def make_split(num_samples: int, name: str) -> FeatureSequenceDataset:
        samples = []
        for _ in range(num_samples):
            verb = int(rng.integers(0, num_verbs))
            noun = int((verb * 3 + rng.integers(0, 7)) % num_nouns)
            length = int(rng.integers(max(4, cfg.max_len // 3), cfg.max_len * 2 + 1))
            base = 0.6 * verb_proto[verb] + 0.45 * noun_proto[noun] + action_bias[verb, noun]
            trend = np.linspace(-0.2, 0.2, length, dtype=np.float32)[:, None] * rng.normal(size=(1, feature_dim)).astype(np.float32)
            noise = rng.normal(scale=0.7, size=(length, feature_dim)).astype(np.float32)
            features = base[None, :] + trend + noise
            samples.append({"features": features.astype(np.float32), "verb": verb, "noun": noun})
        return FeatureSequenceDataset(samples, name=name, feature_dim=feature_dim)

    train = make_split(cfg.synthetic_train_samples, "synthetic-train")
    val = make_split(cfg.synthetic_val_samples, "synthetic-val")
    train = cap_dataset(train, cfg.max_train_samples, cfg.seed)
    val = cap_dataset(val, cfg.max_val_samples, cfg.seed + 1)
    return train, val, num_verbs, num_nouns, feature_dim, "synthetic"


loaded = load_real_feature_dataset(DISCOVERED_FILES, CFG)
if loaded is None:
    train_dataset, val_dataset, NUM_VERBS, NUM_NOUNS, FEATURE_DIM, DATASET_MODE = make_synthetic_datasets(CFG)
else:
    train_dataset, val_dataset, NUM_VERBS, NUM_NOUNS, FEATURE_DIM, DATASET_MODE = loaded

print(f"Dataset mode: {DATASET_MODE}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Classes: verbs={NUM_VERBS}, nouns={NUM_NOUNS}, feature_dim={FEATURE_DIM}")

sample_features, sample_verb, sample_noun = train_dataset[0]
print(f"Sample feature shape before collate: {tuple(sample_features.shape)}")
print(f"Sample labels: verb={int(sample_verb)}, noun={int(sample_noun)}")


def token_float(value: float) -> str:
    text = f"{value:.0e}" if (0 < abs(value) < 1e-3 or abs(value) >= 1e3) else f"{value:g}"
    return text.replace("-", "m").replace("+", "").replace(".", "p")


def token_optional_int(value: Optional[int], prefix: str) -> str:
    return f"{prefix}full" if value is None else f"{prefix}{value}"


def sanitize_token(text: str) -> str:
    allowed = []
    for char in text.lower():
        if char.isalnum() or char in {"-", "_"}:
            allowed.append(char)
        else:
            allowed.append("-")
    token = "".join(allowed).strip("-_")
    while "--" in token:
        token = token.replace("--", "-")
    return token[:60]


def make_run_stem(
    cfg: Config,
    dataset_mode: str,
    feature_dim: int,
    num_verbs: int,
    num_nouns: int,
) -> str:
    if cfg.models_to_run is not None:
        model_tag = "-".join(sanitize_token(model) for model in cfg.models_to_run)
    else:
        model_tag = "mp-gru-rmt" if cfg.run_tiny_rmt else "mp-gru"
    if cfg.sweep_seeds:
        seed_tag = "seeds" + "-".join(str(seed) for seed in cfg.sweep_seeds)
    else:
        seed_tag = f"s{cfg.seed}"
    mode_tag = "dev" if cfg.fast_dev_run else dataset_mode
    parts = [
        mode_tag,
        model_tag,
        f"h{cfg.hidden_dim}",
        f"l{cfg.max_len}",
        f"b{cfg.batch_size}",
        f"e{cfg.epochs}",
        f"lr{token_float(cfg.lr)}",
        f"wd{token_float(cfg.weight_decay)}",
        f"do{token_float(cfg.dropout)}",
        seed_tag,
        f"p{cfg.early_stop_patience}",
        f"mep{cfg.early_stop_min_epochs}",
        token_optional_int(cfg.max_train_samples, "tr"),
        token_optional_int(cfg.max_val_samples, "va"),
        f"d{feature_dim}",
        f"v{num_verbs}",
        f"n{num_nouns}",
    ]
    if cfg.lr_scheduler != "none":
        parts.insert(8, f"sch{sanitize_token(cfg.lr_scheduler)}")
        parts.insert(9, f"eta{token_float(cfg.min_lr_ratio)}")
    if cfg.action_loss_weight != 0.0:
        parts.insert(10, f"alw{token_float(cfg.action_loss_weight)}")
    if cfg.action_score_weight != 0.0:
        parts.insert(11, f"asw{token_float(cfg.action_score_weight)}")
    if cfg.action_score_weights:
        weights = "-".join(token_float(weight) for weight in cfg.action_score_weights)
        parts.insert(12, f"asws{weights}")
    if cfg.run_name:
        parts.insert(0, sanitize_token(cfg.run_name))
    return "_".join(parts)


def create_unique_run_paths(
    cfg: Config,
    dataset_mode: str,
    feature_dim: int,
    num_verbs: int,
    num_nouns: int,
) -> Tuple[str, Path, Path, Path, Path]:
    stem = make_run_stem(cfg, dataset_mode, feature_dim, num_verbs, num_nouns)
    runs_root = OUTPUT_DIR / "runs"
    runs_root.mkdir(parents=True, exist_ok=True)

    run_id = stem
    run_dir = runs_root / run_id
    if run_dir.exists():
        for idx in range(2, 1000):
            candidate_id = f"{stem}_r{idx:02d}"
            candidate_dir = runs_root / candidate_id
            if not candidate_dir.exists():
                run_id = candidate_id
                run_dir = candidate_dir
                break

    checkpoints_dir = run_dir / "checkpoints"
    results_path = run_dir / "results.csv"
    plot_path = run_dir / "quality_efficiency.png"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    return run_id, run_dir, checkpoints_dir, results_path, plot_path


RUN_ID, RUN_DIR, CHECKPOINT_DIR, RESULTS_PATH, PLOT_PATH = create_unique_run_paths(
    CFG,
    DATASET_MODE,
    FEATURE_DIM,
    NUM_VERBS,
    NUM_NOUNS,
)
print(f"Run ID: {RUN_ID}")
print(f"Run directory: {RUN_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")




## Model Definitions



In [ ]:
class MeanPoolMLP(nn.Module):
    def __init__(self, feature_dim: int, num_verbs: int, num_nouns: int, hidden_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.verb_head = nn.Linear(hidden_dim, num_verbs)
        self.noun_head = nn.Linear(hidden_dim, num_nouns)

    def forward(self, features: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        if mask is None:
            pooled = features.mean(dim=1)
        else:
            weights = mask.unsqueeze(-1).to(features.dtype)
            pooled = (features * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)
        hidden = self.encoder(pooled)
        return self.verb_head(hidden), self.noun_head(hidden)


class GRUModel(nn.Module):
    def __init__(self, feature_dim: int, num_verbs: int, num_nouns: int, hidden_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(feature_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.verb_head = nn.Linear(hidden_dim, num_verbs)
        self.noun_head = nn.Linear(hidden_dim, num_nouns)

    def forward(self, features: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        if mask is not None:
            lengths = mask.sum(dim=1).clamp_min(1).detach().cpu()
            packed = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=False)
            _, hidden = self.gru(packed)
        else:
            _, hidden = self.gru(features)
        hidden = self.dropout(hidden[-1])
        return self.verb_head(hidden), self.noun_head(hidden)


class AttentiveGRU(nn.Module):
    def __init__(self, feature_dim: int, num_verbs: int, num_nouns: int, hidden_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(feature_dim, hidden_dim, batch_first=True)
        attn_hidden = max(64, hidden_dim // 2)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, attn_hidden),
            nn.Tanh(),
            nn.Linear(attn_hidden, 1),
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.verb_head = nn.Linear(hidden_dim, num_verbs)
        self.noun_head = nn.Linear(hidden_dim, num_nouns)

    def forward(self, features: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        if mask is not None:
            mask = mask.bool()
            lengths = mask.sum(dim=1).clamp_min(1).detach().cpu()
            packed = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=False)
            packed_outputs, hidden = self.gru(packed)
            outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True, total_length=features.shape[1])
        else:
            outputs, hidden = self.gru(features)
            mask = torch.ones(features.shape[:2], dtype=torch.bool, device=features.device)

        attention_scores = self.attention(outputs).squeeze(-1)
        attention_scores = attention_scores.masked_fill(~mask, torch.finfo(attention_scores.dtype).min)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)
        attended = (outputs * attention_weights).sum(dim=1)
        last_hidden = hidden[-1]
        hidden = self.fusion(torch.cat([last_hidden, attended], dim=1))
        return self.verb_head(hidden), self.noun_head(hidden)


class JointGRUModel(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        num_verbs: int,
        num_nouns: int,
        num_actions: int,
        hidden_dim: int = 256,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.gru = nn.GRU(feature_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.verb_head = nn.Linear(hidden_dim, num_verbs)
        self.noun_head = nn.Linear(hidden_dim, num_nouns)
        self.action_head = nn.Linear(hidden_dim, num_actions)

    def forward(
        self,
        features: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if mask is not None:
            lengths = mask.sum(dim=1).clamp_min(1).detach().cpu()
            packed = pack_padded_sequence(features, lengths, batch_first=True, enforce_sorted=False)
            _, hidden = self.gru(packed)
        else:
            _, hidden = self.gru(features)
        hidden = self.dropout(hidden[-1])
        return self.verb_head(hidden), self.noun_head(hidden), self.action_head(hidden)


class TinyRMT(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        num_verbs: int,
        num_nouns: int,
        max_len: int,
        hidden_dim: int = 256,
        dropout: float = 0.2,
        memory_tokens: int = 2,
        num_layers: int = 1,
        num_heads: int = 4,
    ):
        super().__init__()
        self.memory_tokens = memory_tokens
        self.input_proj = nn.Linear(feature_dim, hidden_dim)
        self.memory = nn.Parameter(torch.randn(1, memory_tokens, hidden_dim) * 0.02)
        self.pos = nn.Parameter(torch.zeros(1, memory_tokens + max_len, hidden_dim))
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.verb_head = nn.Linear(hidden_dim, num_verbs)
        self.noun_head = nn.Linear(hidden_dim, num_nouns)

    def forward(self, features: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size, seq_len, _ = features.shape
        projected = self.input_proj(features)
        memory = self.memory.expand(batch_size, -1, -1)
        tokens = torch.cat([memory, projected], dim=1)
        tokens = tokens + self.pos[:, : tokens.shape[1]]

        key_padding_mask = None
        if mask is not None:
            memory_mask = torch.zeros(batch_size, self.memory_tokens, dtype=torch.bool, device=mask.device)
            key_padding_mask = torch.cat([memory_mask, ~mask.bool()], dim=1)

        encoded = self.encoder(tokens, src_key_padding_mask=key_padding_mask)
        hidden = self.norm(encoded[:, : self.memory_tokens].mean(dim=1))
        hidden = self.dropout(hidden)
        return self.verb_head(hidden), self.noun_head(hidden)


def count_trainable_params(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)




## Semantic Verb-Noun Prior



In [ ]:
def dataset_labels(dataset: Dataset) -> List[Tuple[int, int]]:
    if isinstance(dataset, Subset):
        base_labels = dataset_labels(dataset.dataset)
        return [base_labels[i] for i in dataset.indices]
    if hasattr(dataset, "iter_labels"):
        return [(int(v), int(n)) for v, n in dataset.iter_labels()]
    labels = []
    for idx in range(len(dataset)):
        _, verb, noun = dataset[idx]
        labels.append((int(verb), int(noun)))
    return labels


def compute_log_prior(dataset: Dataset, num_verbs: int, num_nouns: int, alpha: float = 1.0) -> torch.Tensor:
    counts = torch.full((num_verbs, num_nouns), float(alpha), dtype=torch.float32)
    for verb, noun in dataset_labels(dataset):
        if 0 <= verb < num_verbs and 0 <= noun < num_nouns:
            counts[verb, noun] += 1.0
    probs = counts / counts.sum()
    log_prior = probs.log()
    print(f"Prior shape: {tuple(log_prior.shape)}, alpha={alpha}")
    return log_prior


def build_action_index_table(
    dataset: Dataset,
    num_verbs: int,
    num_nouns: int,
) -> Tuple[torch.Tensor, List[Tuple[int, int]]]:
    pairs = sorted(set(dataset_labels(dataset)))
    table = torch.full((num_verbs, num_nouns), -1, dtype=torch.long)
    action_pairs = []
    for action_idx, (verb, noun) in enumerate(pairs):
        if 0 <= verb < num_verbs and 0 <= noun < num_nouns:
            table[verb, noun] = action_idx
            action_pairs.append((verb, noun))
    print(f"Seen train action pairs: {len(action_pairs)} / {num_verbs * num_nouns}")
    return table, action_pairs


LOG_PRIOR = compute_log_prior(train_dataset, NUM_VERBS, NUM_NOUNS, alpha=CFG.prior_alpha)
ACTION_INDEX_TABLE, ACTION_PAIRS = build_action_index_table(train_dataset, NUM_VERBS, NUM_NOUNS)
NUM_ACTIONS = len(ACTION_PAIRS)
ACTION_FLAT_INDICES = torch.tensor(
    [verb * NUM_NOUNS + noun for verb, noun in ACTION_PAIRS],
    dtype=torch.long,
)




## Metrics



In [ ]:
def topk_correct(logits: torch.Tensor, targets: torch.Tensor, k: int) -> int:
    k = min(k, logits.shape[1])
    pred = logits.topk(k, dim=1).indices
    return pred.eq(targets.unsqueeze(1)).any(dim=1).sum().item()


def joint_topk_correct(
    verb_logits: torch.Tensor,
    noun_logits: torch.Tensor,
    verb_targets: torch.Tensor,
    noun_targets: torch.Tensor,
    log_prior: torch.Tensor,
    prior_lambda: float,
    k: int,
    action_logits: Optional[torch.Tensor] = None,
    action_flat_indices: Optional[torch.Tensor] = None,
    action_score_weight: float = 0.0,
) -> int:
    num_nouns = noun_logits.shape[1]
    joint_scores = verb_logits.unsqueeze(2) + noun_logits.unsqueeze(1)
    if prior_lambda != 0.0:
        joint_scores = joint_scores + prior_lambda * log_prior.to(joint_scores.device).unsqueeze(0)
    flat_scores = joint_scores.flatten(start_dim=1)
    if action_logits is not None and action_flat_indices is not None and action_score_weight != 0.0:
        flat_indices = action_flat_indices.to(flat_scores.device)
        if flat_indices.numel() > 0:
            if action_logits.shape[1] != flat_indices.numel():
                raise ValueError(
                    f"action_logits has {action_logits.shape[1]} columns, "
                    f"but action_flat_indices has {flat_indices.numel()} entries"
                )
            bonus = torch.zeros_like(flat_scores)
            expanded_indices = flat_indices.unsqueeze(0).expand(action_logits.shape[0], -1)
            bonus.scatter_add_(1, expanded_indices, action_score_weight * action_logits.to(flat_scores.dtype))
            flat_scores = flat_scores + bonus
    k = min(k, flat_scores.shape[1])
    topk = flat_scores.topk(k, dim=1).indices
    target_flat = verb_targets * num_nouns + noun_targets
    return topk.eq(target_flat.unsqueeze(1)).any(dim=1).sum().item()


def unpack_model_outputs(outputs: Any) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    if isinstance(outputs, tuple) and len(outputs) == 3:
        return outputs[0], outputs[1], outputs[2]
    if isinstance(outputs, tuple) and len(outputs) == 2:
        return outputs[0], outputs[1], None
    raise ValueError("Model forward must return (verb_logits, noun_logits) or (verb_logits, noun_logits, action_logits)")


def batch_action_targets(
    verb_targets: torch.Tensor,
    noun_targets: torch.Tensor,
    action_index_table: torch.Tensor,
) -> torch.Tensor:
    table = action_index_table.to(verb_targets.device)
    return table[verb_targets, noun_targets]


def classification_loss(
    verb_logits: torch.Tensor,
    noun_logits: torch.Tensor,
    verb_targets: torch.Tensor,
    noun_targets: torch.Tensor,
    action_logits: Optional[torch.Tensor] = None,
    action_targets: Optional[torch.Tensor] = None,
    action_loss_weight: float = 0.0,
) -> torch.Tensor:
    loss = F.cross_entropy(verb_logits, verb_targets) + F.cross_entropy(noun_logits, noun_targets)
    if action_logits is not None and action_targets is not None and action_loss_weight != 0.0:
        valid = action_targets >= 0
        if valid.any():
            loss = loss + action_loss_weight * F.cross_entropy(action_logits[valid], action_targets[valid])
    return loss


def empty_metric_sums() -> Dict[str, float]:
    return {
        "loss_sum": 0.0,
        "verb_top1": 0.0,
        "verb_top5": 0.0,
        "noun_top1": 0.0,
        "noun_top5": 0.0,
        "action_top1": 0.0,
        "action_top5": 0.0,
        "samples": 0.0,
    }


def finalize_metrics(metric_sums: Dict[str, float]) -> Dict[str, float]:
    samples = max(1.0, metric_sums["samples"])
    return {
        "loss": metric_sums["loss_sum"] / samples,
        "verb_top1": metric_sums["verb_top1"] / samples,
        "verb_top5": metric_sums["verb_top5"] / samples,
        "noun_top1": metric_sums["noun_top1"] / samples,
        "noun_top5": metric_sums["noun_top5"] / samples,
        "action_top1": metric_sums["action_top1"] / samples,
        "action_top5": metric_sums["action_top5"] / samples,
    }




## Training and Evaluation Loops



In [ ]:
def autocast_context(device: torch.device):
    enabled = device.type == "cuda"
    if hasattr(torch, "amp"):
        return torch.amp.autocast(device_type="cuda", enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


def make_grad_scaler(device: torch.device):
    enabled = device.type == "cuda"
    if hasattr(torch, "amp"):
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except TypeError:
            pass
    return torch.cuda.amp.GradScaler(enabled=enabled)


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    device: torch.device,
    cfg: Config,
) -> Tuple[float, float]:
    model.train()
    loss_sum = 0.0
    samples = 0
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            verb_logits, noun_logits, action_logits = unpack_model_outputs(model(batch["features"], batch["mask"]))
            action_targets = batch_action_targets(batch["verb"], batch["noun"], ACTION_INDEX_TABLE)
            loss = classification_loss(
                verb_logits,
                noun_logits,
                batch["verb"],
                batch["noun"],
                action_logits=action_logits,
                action_targets=action_targets,
                action_loss_weight=cfg.action_loss_weight,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        batch_size = int(batch["verb"].shape[0])
        loss_sum += float(loss.detach().cpu()) * batch_size
        samples += batch_size

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return loss_sum / max(1, samples), elapsed


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    log_prior: torch.Tensor,
    prior_lambda: float,
    cfg: Config,
    action_score_weight: Optional[float] = None,
) -> Dict[str, float]:
    model.eval()
    sums = empty_metric_sums()
    score_weight = cfg.action_score_weight if action_score_weight is None else float(action_score_weight)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        with autocast_context(device):
            verb_logits, noun_logits, action_logits = unpack_model_outputs(model(batch["features"], batch["mask"]))
            action_targets = batch_action_targets(batch["verb"], batch["noun"], ACTION_INDEX_TABLE)
            loss = classification_loss(
                verb_logits,
                noun_logits,
                batch["verb"],
                batch["noun"],
                action_logits=action_logits,
                action_targets=action_targets,
                action_loss_weight=cfg.action_loss_weight,
            )

        batch_size = int(batch["verb"].shape[0])
        sums["loss_sum"] += float(loss.detach().cpu()) * batch_size
        sums["verb_top1"] += topk_correct(verb_logits, batch["verb"], 1)
        sums["verb_top5"] += topk_correct(verb_logits, batch["verb"], 5)
        sums["noun_top1"] += topk_correct(noun_logits, batch["noun"], 1)
        sums["noun_top5"] += topk_correct(noun_logits, batch["noun"], 5)
        sums["action_top1"] += joint_topk_correct(
            verb_logits,
            noun_logits,
            batch["verb"],
            batch["noun"],
            log_prior,
            prior_lambda,
            1,
            action_logits=action_logits,
            action_flat_indices=ACTION_FLAT_INDICES,
            action_score_weight=score_weight,
        )
        sums["action_top5"] += joint_topk_correct(
            verb_logits,
            noun_logits,
            batch["verb"],
            batch["noun"],
            log_prior,
            prior_lambda,
            5,
            action_logits=action_logits,
            action_flat_indices=ACTION_FLAT_INDICES,
            action_score_weight=score_weight,
        )
        sums["samples"] += batch_size

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    metrics = finalize_metrics(sums)
    metrics["inference_ms_per_sample"] = 1000.0 * elapsed / max(1.0, sums["samples"])
    metrics["peak_gpu_mem_mb"] = (
        torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else 0.0
    )
    return metrics


@torch.no_grad()
def evaluate_prior_only(
    loader: DataLoader,
    device: torch.device,
    log_prior: torch.Tensor,
) -> Dict[str, float]:
    """Evaluate a no-training statistical baseline from train-set label frequencies."""
    sums = empty_metric_sums()
    log_prior_device = log_prior.to(device)
    verb_scores_base = torch.logsumexp(log_prior_device, dim=1)
    noun_scores_base = torch.logsumexp(log_prior_device, dim=0)
    flat_prior_scores = log_prior_device.flatten()
    num_nouns = log_prior_device.shape[1]

    top1_flat = flat_prior_scores.topk(min(1, flat_prior_scores.numel())).indices
    top5_flat = flat_prior_scores.topk(min(5, flat_prior_scores.numel())).indices

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        batch_size = int(batch["verb"].shape[0])
        verb_scores = verb_scores_base.unsqueeze(0).expand(batch_size, -1)
        noun_scores = noun_scores_base.unsqueeze(0).expand(batch_size, -1)
        target_flat = batch["verb"] * num_nouns + batch["noun"]
        loss = F.nll_loss(verb_scores, batch["verb"], reduction="sum") + F.nll_loss(
            noun_scores,
            batch["noun"],
            reduction="sum",
        )

        sums["loss_sum"] += float(loss.detach().cpu())
        sums["verb_top1"] += topk_correct(verb_scores, batch["verb"], 1)
        sums["verb_top5"] += topk_correct(verb_scores, batch["verb"], 5)
        sums["noun_top1"] += topk_correct(noun_scores, batch["noun"], 1)
        sums["noun_top5"] += topk_correct(noun_scores, batch["noun"], 5)
        sums["action_top1"] += top1_flat.unsqueeze(0).eq(target_flat.unsqueeze(1)).any(dim=1).sum().item()
        sums["action_top5"] += top5_flat.unsqueeze(0).eq(target_flat.unsqueeze(1)).any(dim=1).sum().item()
        sums["samples"] += batch_size

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    metrics = finalize_metrics(sums)
    metrics["inference_ms_per_sample"] = 1000.0 * elapsed / max(1.0, sums["samples"])
    metrics["peak_gpu_mem_mb"] = (
        torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else 0.0
    )
    return metrics


def make_loaders(
    train_ds: Dataset,
    val_ds: Dataset,
    cfg: Config,
    device: torch.device,
) -> Tuple[DataLoader, DataLoader]:
    collate_fn = make_collate_fn(cfg.max_len, cfg.truncate_mode)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        collate_fn=collate_fn,
    )
    return train_loader, val_loader


TRAIN_HISTORY: List[Dict[str, Any]] = []


def make_scheduler(
    optimizer: torch.optim.Optimizer,
    cfg: Config,
) -> Optional[torch.optim.lr_scheduler.LRScheduler]:
    scheduler_name = cfg.lr_scheduler.lower().strip()
    if scheduler_name in {"", "none"}:
        return None
    if scheduler_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(1, cfg.epochs),
            eta_min=cfg.lr * cfg.min_lr_ratio,
        )
    raise ValueError(f"Unknown lr_scheduler={cfg.lr_scheduler!r}; expected 'none' or 'cosine'")


def train_model(
    model_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    log_prior: torch.Tensor,
    cfg: Config,
) -> Tuple[nn.Module, float]:
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = make_scheduler(optimizer, cfg)
    scaler = make_grad_scaler(device)
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}_best.pt"
    best_action_top5 = -1.0
    best_epoch = -1
    train_times: List[float] = []
    epochs_without_improvement = 0

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_seconds = train_one_epoch(model, train_loader, optimizer, scaler, device, cfg)
        train_times.append(train_seconds)
        val_metrics = evaluate_model(model, val_loader, device, log_prior, prior_lambda=0.0, cfg=cfg)
        score = val_metrics["action_top5"]
        current_lr = float(optimizer.param_groups[0]["lr"])
        TRAIN_HISTORY.append(
            {
                "run_id": RUN_ID,
                "model": model_name,
                "epoch": int(epoch),
                "lr": current_lr,
                "train_loss": float(train_loss),
                "train_time_sec": float(train_seconds),
                "val_loss": float(val_metrics["loss"]),
                "val_verb_top1": float(val_metrics["verb_top1"]),
                "val_verb_top5": float(val_metrics["verb_top5"]),
                "val_noun_top1": float(val_metrics["noun_top1"]),
                "val_noun_top5": float(val_metrics["noun_top5"]),
                "val_action_top1": float(val_metrics["action_top1"]),
                "val_action_top5_no_prior": float(val_metrics["action_top5"]),
            }
        )
        print(
            f"[{model_name}] epoch {epoch}/{cfg.epochs} "
            f"train_loss={train_loss:.4f} val_action_top5={score:.4f} "
            f"train_sec={train_seconds:.2f} lr={current_lr:.2e}"
        )

        if score > best_action_top5:
            best_action_top5 = score
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epoch >= cfg.early_stop_min_epochs and epochs_without_improvement >= cfg.early_stop_patience:
                print(f"[{model_name}] early stopping at epoch {epoch}")
                break
        if scheduler is not None:
            scheduler.step()

    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"[{model_name}] best epoch={best_epoch}, best val action_top5 without prior={best_action_top5:.4f}")
    return model, float(np.mean(train_times)) if train_times else 0.0




## Experiment Runner



In [ ]:
def build_model_factories(
    feature_dim: int,
    num_verbs: int,
    num_nouns: int,
    num_actions: int,
    cfg: Config,
) -> Dict[str, Any]:
    factories: Dict[str, Any] = {
        "prior_only": None,
        "meanpool_mlp": lambda: MeanPoolMLP(feature_dim, num_verbs, num_nouns, cfg.hidden_dim, cfg.dropout),
        "gru": lambda: GRUModel(feature_dim, num_verbs, num_nouns, cfg.hidden_dim, cfg.dropout),
        "attn_gru": lambda: AttentiveGRU(feature_dim, num_verbs, num_nouns, cfg.hidden_dim, cfg.dropout),
        "joint_gru": lambda: JointGRUModel(
            feature_dim,
            num_verbs,
            num_nouns,
            num_actions,
            cfg.hidden_dim,
            cfg.dropout,
        ),
        "tiny_rmt": lambda: TinyRMT(
            feature_dim,
            num_verbs,
            num_nouns,
            max_len=cfg.max_len,
            hidden_dim=cfg.hidden_dim,
            dropout=cfg.dropout,
        ),
    }
    selected = cfg.models_to_run
    if selected is None:
        selected = ("meanpool_mlp", "gru", "tiny_rmt") if cfg.run_tiny_rmt else ("meanpool_mlp", "gru")

    unknown = [model_name for model_name in selected if model_name not in factories]
    if unknown:
        raise ValueError(f"Unknown models_to_run entries: {unknown}. Available: {sorted(factories)}")
    return {model_name: factories[model_name] for model_name in selected}


def save_results_table(rows: List[Dict[str, Any]], output_path: Path) -> Any:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if pd is not None:
        frame = pd.DataFrame(rows)
        frame.to_csv(output_path, index=False)
        return frame

    if not rows:
        return rows
    with output_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return rows


def rows_from_results(results: Any) -> List[Dict[str, Any]]:
    if results is None:
        return []
    if pd is not None and hasattr(results, "to_dict"):
        return list(results.to_dict(orient="records"))
    if isinstance(results, list):
        return [dict(row) for row in results]
    return []


def experiment_seeds(cfg: Config) -> Tuple[int, ...]:
    if cfg.sweep_seeds is not None:
        if not cfg.sweep_seeds:
            raise ValueError("sweep_seeds must be None or a non-empty tuple of integers")
        return tuple(int(seed) for seed in cfg.sweep_seeds)
    return (int(cfg.seed),)


def evaluation_action_score_weights(cfg: Config) -> Tuple[float, ...]:
    if cfg.action_score_weights is not None:
        if not cfg.action_score_weights:
            raise ValueError("action_score_weights must be None or a non-empty tuple of floats")
        return tuple(float(weight) for weight in cfg.action_score_weights)
    return (float(cfg.action_score_weight),)


def aggregate_seed_results(results: Any) -> Any:
    rows = rows_from_results(results)
    if not rows:
        return [] if pd is None else pd.DataFrame()

    group_keys = ["model", "prior_lambda", "action_loss_weight", "action_score_weight"]
    metric_keys = [
        "verb_top1",
        "verb_top5",
        "noun_top1",
        "noun_top5",
        "action_top1",
        "action_top5",
        "train_time_per_epoch_sec",
        "inference_ms_per_sample",
        "peak_gpu_mem_mb",
    ]
    static_keys = ["params", "num_actions"]

    if pd is not None:
        frame = pd.DataFrame(rows)
        grouped = frame.groupby(group_keys, as_index=False)
        aggregate = grouped[metric_keys].agg(["mean", "std"]).reset_index()
        aggregate.columns = [
            "_".join(str(part) for part in column if part)
            if isinstance(column, tuple)
            else str(column)
            for column in aggregate.columns
        ]
        params = grouped[static_keys].first().reset_index()
        aggregate = aggregate.merge(params, on=group_keys, how="left")
        aggregate["run_id"] = RUN_ID
        aggregate["num_seeds"] = len(set(frame["seed"])) if "seed" in frame.columns else 1
        preferred = [
            "run_id",
            "model",
            "prior_lambda",
            "action_loss_weight",
            "action_score_weight",
            "params",
            "num_actions",
            "num_seeds",
        ]
        remaining = [column for column in aggregate.columns if column not in preferred]
        return aggregate[preferred + remaining]

    groups: Dict[Tuple[Any, ...], List[Dict[str, Any]]] = {}
    for row in rows:
        key = tuple(row[group_key] for group_key in group_keys)
        groups.setdefault(key, []).append(row)

    aggregate_rows = []
    for key, group in groups.items():
        out = {
            "run_id": RUN_ID,
            "model": key[0],
            "prior_lambda": key[1],
            "action_loss_weight": key[2],
            "action_score_weight": key[3],
            "params": group[0]["params"],
            "num_actions": group[0].get("num_actions", 0),
            "num_seeds": len({row.get("seed") for row in group}),
        }
        for metric in metric_keys:
            values = np.asarray([float(row[metric]) for row in group], dtype=np.float64)
            out[f"{metric}_mean"] = float(values.mean())
            out[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        aggregate_rows.append(out)
    return sorted(aggregate_rows, key=lambda row: (-row["action_top5_mean"], row["inference_ms_per_sample_mean"]))


def plot_quality_efficiency(results: Any, output_path: Path) -> None:
    rows = rows_from_results(results)
    if plt is None or not rows:
        print("[plot] matplotlib unavailable, or no results; skipping plot")
        return
    model_names = sorted({row["model"] for row in rows})
    plt.figure(figsize=(8, 5))
    for model_name in model_names:
        group = [row for row in rows if row["model"] == model_name]
        xs = [row["inference_ms_per_sample"] for row in group]
        ys = [row["action_top5"] for row in group]
        plt.scatter(xs, ys, label=model_name, s=55)
        for row in group:
            plt.annotate(f"l={row['prior_lambda']}", (row["inference_ms_per_sample"], row["action_top5"]), fontsize=8)
    plt.xlabel("Validation inference time, ms/sample")
    plt.ylabel("Joint action top-5 accuracy")
    plt.title("Quality vs efficiency")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()
    print(f"[plot] saved {output_path}")


def plot_training_history(history_rows: List[Dict[str, Any]], output_path: Path) -> None:
    if plt is None or not history_rows:
        print("[plot] matplotlib unavailable, or no training history; skipping learning curves")
        return

    model_names = sorted({str(row["model"]) for row in history_rows})
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for model_name in model_names:
        group = sorted([row for row in history_rows if row["model"] == model_name], key=lambda row: int(row["epoch"]))
        epochs = [int(row["epoch"]) for row in group]
        axes[0].plot(epochs, [float(row["train_loss"]) for row in group], label=model_name)
        axes[1].plot(epochs, [float(row["val_loss"]) for row in group], label=model_name)
        axes[2].plot(epochs, [float(row["val_action_top5_no_prior"]) for row in group], label=model_name)

    axes[0].set_title("Training loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Validation loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[2].set_title("Validation action top-5, no prior")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Accuracy")
    for axis in axes:
        axis.grid(True, alpha=0.25)
    axes[2].legend(fontsize=8)
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=160)
    plt.close(fig)
    print(f"[plot] saved learning curves: {output_path}")


def json_safe_config(cfg: Config) -> Dict[str, Any]:
    values = asdict(cfg)
    for key, value in list(values.items()):
        if isinstance(value, tuple):
            values[key] = list(value)
    return values


def save_run_metadata(results: Any, output_path: Path) -> None:
    rows = rows_from_results(results)
    metadata = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "dataset_mode": DATASET_MODE,
        "dataset": {
            "train_samples": len(train_dataset),
            "val_samples": len(val_dataset),
            "num_verbs": NUM_VERBS,
            "num_nouns": NUM_NOUNS,
            "num_seen_train_actions": NUM_ACTIONS,
            "feature_dim": FEATURE_DIM,
            "sample_shape_before_collate": list(sample_features.shape),
        },
        "config": json_safe_config(CFG),
        "outputs": {
            "results_csv": str(RESULTS_PATH),
            "flat_results_csv": str(OUTPUT_DIR / f"{RUN_ID}_results.csv"),
            "aggregate_csv": str(RUN_DIR / "aggregate_results.csv"),
            "flat_aggregate_csv": str(OUTPUT_DIR / f"{RUN_ID}_aggregate_results.csv"),
            "training_history_csv": str(RUN_DIR / "training_history.csv"),
            "training_curves": str(RUN_DIR / "training_curves.png"),
            "plot": str(PLOT_PATH),
            "checkpoints": str(CHECKPOINT_DIR),
        },
        "best_action_top5": max(rows, key=lambda row: row["action_top5"]) if rows else None,
    }
    output_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"[metadata] saved {output_path}")


def run_all_experiments(cfg: Config) -> Any:
    TRAIN_HISTORY.clear()
    factories = build_model_factories(FEATURE_DIM, NUM_VERBS, NUM_NOUNS, NUM_ACTIONS, cfg)
    results: List[Dict[str, Any]] = []
    seeds = experiment_seeds(cfg)
    score_weights = evaluation_action_score_weights(cfg)
    prior_only_done = False
    print(f"Experiment seeds: {seeds}")
    print(f"Evaluation action-score weights: {score_weights}")

    for seed in seeds:
        print("=" * 80)
        print(f"Starting seed {seed}")
        for model_name, factory in factories.items():
            set_seed(seed)
            train_loader, val_loader = make_loaders(train_dataset, val_dataset, cfg, DEVICE)
            if model_name == "prior_only":
                if prior_only_done:
                    continue
                prior_only_done = True
                print("=" * 80)
                print("Evaluating prior_only baseline: train-set verb/noun/action frequency, no training")
                metrics = evaluate_prior_only(val_loader, DEVICE, LOG_PRIOR)
                row = {
                    "run_id": RUN_ID,
                    "dataset_mode": DATASET_MODE,
                    "seed": int(seed),
                    "model": model_name,
                    "prior_lambda": 1.0,
                    "action_loss_weight": 0.0,
                    "action_score_weight": 0.0,
                    "params": 0,
                    "num_actions": int(NUM_ACTIONS),
                    "train_time_per_epoch_sec": 0.0,
                    "inference_ms_per_sample": float(metrics["inference_ms_per_sample"]),
                    "peak_gpu_mem_mb": float(metrics["peak_gpu_mem_mb"]),
                    "verb_top1": float(metrics["verb_top1"]),
                    "verb_top5": float(metrics["verb_top5"]),
                    "noun_top1": float(metrics["noun_top1"]),
                    "noun_top5": float(metrics["noun_top5"]),
                    "action_top1": float(metrics["action_top1"]),
                    "action_top5": float(metrics["action_top5"]),
                }
                results.append(row)
                print(
                    f"[prior_only] verb@1={row['verb_top1']:.3f} noun@1={row['noun_top1']:.3f} "
                    f"action@5={row['action_top5']:.3f} ms/sample={row['inference_ms_per_sample']:.3f}"
                )
                continue

            if factory is None:
                raise RuntimeError(f"Model factory for {model_name} is unexpectedly missing")
            model = factory()
            params = count_trainable_params(model)
            checkpoint_model_name = model_name if len(seeds) == 1 else f"{model_name}_seed{seed}"
            print("=" * 80)
            print(f"Training {model_name} seed={seed}: trainable parameters={params:,}")
            model, train_time_per_epoch = train_model(
                checkpoint_model_name,
                model,
                train_loader,
                val_loader,
                DEVICE,
                LOG_PRIOR,
                cfg,
            )

            for eval_action_score_weight in score_weights:
                for prior_lambda in cfg.prior_lambdas:
                    metrics = evaluate_model(
                        model,
                        val_loader,
                        DEVICE,
                        LOG_PRIOR,
                        prior_lambda=prior_lambda,
                        cfg=cfg,
                        action_score_weight=eval_action_score_weight,
                    )
                    row = {
                        "run_id": RUN_ID,
                        "dataset_mode": DATASET_MODE,
                        "seed": int(seed),
                        "model": model_name,
                        "prior_lambda": float(prior_lambda),
                        "action_loss_weight": float(cfg.action_loss_weight),
                        "action_score_weight": float(eval_action_score_weight),
                        "params": int(params),
                        "num_actions": int(NUM_ACTIONS),
                        "train_time_per_epoch_sec": float(train_time_per_epoch),
                        "inference_ms_per_sample": float(metrics["inference_ms_per_sample"]),
                        "peak_gpu_mem_mb": float(metrics["peak_gpu_mem_mb"]),
                        "verb_top1": float(metrics["verb_top1"]),
                        "verb_top5": float(metrics["verb_top5"]),
                        "noun_top1": float(metrics["noun_top1"]),
                        "noun_top5": float(metrics["noun_top5"]),
                        "action_top1": float(metrics["action_top1"]),
                        "action_top5": float(metrics["action_top5"]),
                    }
                    results.append(row)
                    print(
                        f"[{model_name}] seed={seed} lambda={prior_lambda:.1f} "
                        f"action_w={eval_action_score_weight:.2f} "
                        f"verb@1={row['verb_top1']:.3f} noun@1={row['noun_top1']:.3f} "
                        f"action@5={row['action_top5']:.3f} ms/sample={row['inference_ms_per_sample']:.3f}"
                    )

    results_frame = save_results_table(results, RESULTS_PATH)
    print(f"[results] saved run-specific CSV: {RESULTS_PATH}")

    aggregate_results = aggregate_seed_results(results_frame)
    aggregate_path = RUN_DIR / "aggregate_results.csv"
    save_results_table(rows_from_results(aggregate_results), aggregate_path)
    print(f"[results] saved aggregate CSV: {aggregate_path}")

    flat_results_path = OUTPUT_DIR / f"{RUN_ID}_results.csv"
    save_results_table(results, flat_results_path)
    print(f"[results] saved flat unique CSV: {flat_results_path}")

    flat_aggregate_path = OUTPUT_DIR / f"{RUN_ID}_aggregate_results.csv"
    save_results_table(rows_from_results(aggregate_results), flat_aggregate_path)
    print(f"[results] saved flat aggregate CSV: {flat_aggregate_path}")

    if cfg.save_latest_copy:
        latest_results_path = OUTPUT_DIR / "latest_results_rgb_features.csv"
        save_results_table(results, latest_results_path)
        print(f"[results] saved latest convenience CSV: {latest_results_path}")
        latest_aggregate_path = OUTPUT_DIR / "latest_aggregate_results_rgb_features.csv"
        save_results_table(rows_from_results(aggregate_results), latest_aggregate_path)
        print(f"[results] saved latest aggregate CSV: {latest_aggregate_path}")

    if TRAIN_HISTORY:
        history_path = RUN_DIR / "training_history.csv"
        save_results_table(TRAIN_HISTORY, history_path)
        print(f"[history] saved training history CSV: {history_path}")

        flat_history_path = OUTPUT_DIR / f"{RUN_ID}_training_history.csv"
        save_results_table(TRAIN_HISTORY, flat_history_path)
        print(f"[history] saved flat training history CSV: {flat_history_path}")

        history_plot_path = RUN_DIR / "training_curves.png"
        plot_training_history(TRAIN_HISTORY, history_plot_path)
        if history_plot_path.exists():
            flat_history_plot_path = OUTPUT_DIR / f"{RUN_ID}_training_curves.png"
            shutil.copy2(history_plot_path, flat_history_plot_path)
            print(f"[plot] saved flat learning curves: {flat_history_plot_path}")

        if cfg.save_latest_copy:
            latest_history_path = OUTPUT_DIR / "latest_training_history_rgb_features.csv"
            save_results_table(TRAIN_HISTORY, latest_history_path)
            print(f"[history] saved latest training history CSV: {latest_history_path}")
            if history_plot_path.exists():
                latest_history_plot_path = OUTPUT_DIR / "latest_training_curves_rgb_features.png"
                shutil.copy2(history_plot_path, latest_history_plot_path)
                print(f"[plot] saved latest learning curves: {latest_history_plot_path}")

    plot_quality_efficiency(results_frame, PLOT_PATH)
    flat_plot_path = OUTPUT_DIR / f"{RUN_ID}_quality_efficiency.png"
    if PLOT_PATH.exists():
        shutil.copy2(PLOT_PATH, flat_plot_path)
        print(f"[plot] saved flat unique plot: {flat_plot_path}")
        if cfg.save_latest_copy:
            latest_plot_path = OUTPUT_DIR / "latest_quality_efficiency.png"
            shutil.copy2(PLOT_PATH, latest_plot_path)
            print(f"[plot] saved latest convenience plot: {latest_plot_path}")

    save_run_metadata(results_frame, RUN_DIR / "metadata.json")
    return results_frame


results_df = run_all_experiments(CFG)
if pd is not None and hasattr(results_df, "sort_values"):
    display_columns = [
        "seed",
        "model",
        "prior_lambda",
        "action_loss_weight",
        "action_score_weight",
        "params",
        "num_actions",
        "train_time_per_epoch_sec",
        "inference_ms_per_sample",
        "verb_top1",
        "verb_top5",
        "noun_top1",
        "noun_top5",
        "action_top1",
        "action_top5",
    ]
    print(results_df[display_columns].sort_values(["action_top5", "inference_ms_per_sample"], ascending=[False, True]))
    aggregate_df = aggregate_seed_results(results_df)
    if hasattr(aggregate_df, "sort_values") and "action_top5_mean" in aggregate_df.columns:
        aggregate_display_columns = [
            "model",
            "prior_lambda",
            "action_loss_weight",
            "action_score_weight",
            "params",
            "num_actions",
            "num_seeds",
            "action_top5_mean",
            "action_top5_std",
            "action_top1_mean",
            "verb_top1_mean",
            "noun_top1_mean",
            "inference_ms_per_sample_mean",
        ]
        print("Aggregate over seeds:")
        print(aggregate_df[aggregate_display_columns].sort_values(["action_top5_mean", "inference_ms_per_sample_mean"], ascending=[False, True]))




## Scope Note: Action Genome, DINOv2, and V-JEPA

Action Genome is not used here because the target task is egocentric
next-action anticipation from temporal RGB features. Action Genome is mainly
scene-graph/action-relation oriented and would require a different data and
model pipeline.

DINOv2 and V-JEPA are relevant as frozen visual encoders, but they are not used
in the main experiments because the current constraints favor already
pre-extracted RGB features. Running whole-dataset feature extraction or training
a visual backbone would not fit the available Kaggle time, RAM, disk, and GPU
budget.



## Final Summary



In [ ]:
def print_final_summary(results: Any, dataset_mode: str) -> None:
    rows = rows_from_results(results)
    if not rows:
        print("No results table is available for automatic summary.")
        return

    eps = 1e-9
    for row in rows:
        row["quality_efficiency"] = row["action_top5"] / max(float(row["inference_ms_per_sample"]), eps)
    best_quality = sorted(rows, key=lambda row: (-row["action_top5"], row["inference_ms_per_sample"]))[0]
    fastest = sorted(rows, key=lambda row: (row["inference_ms_per_sample"], -row["action_top5"]))[0]
    best_tradeoff = sorted(rows, key=lambda row: (-row["quality_efficiency"], -row["action_top5"]))[0]

    print("=" * 80)
    print("Final summary")
    print(f"Run ID: {RUN_ID}")
    print(f"Dataset mode: {dataset_mode}")
    if dataset_mode == "synthetic":
        print("WARNING: synthetic smoke-test mode; do not report these numbers as real accuracy.")
    print(
        "Best action top-5 method: "
        f"{best_quality['model']} with lambda={best_quality['prior_lambda']} "
        f"(action_top5={best_quality['action_top5']:.4f}, "
        f"ms/sample={best_quality['inference_ms_per_sample']:.3f})"
    )
    print(
        "Best efficiency method: "
        f"{fastest['model']} with lambda={fastest['prior_lambda']} "
        f"(ms/sample={fastest['inference_ms_per_sample']:.3f}, "
        f"action_top5={fastest['action_top5']:.4f})"
    )
    print(
        "Best quality/efficiency tradeoff: "
        f"{best_tradeoff['model']} with lambda={best_tradeoff['prior_lambda']} "
        f"(action_top5/ms={best_tradeoff['quality_efficiency']:.4f})"
    )
    print(f"Run directory: {RUN_DIR}")
    print(f"Results CSV: {RESULTS_PATH}")
    print(f"Flat results CSV: {OUTPUT_DIR / f'{RUN_ID}_results.csv'}")
    if PLOT_PATH.exists():
        print(f"Plot: {PLOT_PATH}")
        print(f"Flat plot: {OUTPUT_DIR / f'{RUN_ID}_quality_efficiency.png'}")
    else:
        print("Plot: not created because matplotlib was unavailable or no results were present")
    print(f"Checkpoints: {CHECKPOINT_DIR}")


print_final_summary(results_df, DATASET_MODE)




## Optional Raw-Video Qualitative Demo

This section is for presentation assets only. It does **not** re-extract the
RULSTM RGB features from raw video. Instead, it pairs raw EPIC-KITCHENS
validation clips with the matching pre-extracted feature sample used by the
model. This keeps the quantitative evaluation honest while letting the
presentation show real video frames and top-k predictions.

To run it on Kaggle:

1. Attach a small dataset containing the relevant raw validation videos, e.g.
   `P01_01.mp4`.
2. Keep the trained JointGRU checkpoint in `/kaggle/working/runs/.../checkpoints`
   or set `RAW_VIDEO_DEMO_CHECKPOINT`.
3. Set `RUN_RAW_VIDEO_DEMO = True` in the cell below or set environment
   variable `RUN_RAW_VIDEO_DEMO=1`.



In [ ]:
RUN_RAW_VIDEO_DEMO = env_flag("RUN_RAW_VIDEO_DEMO", True)
RAW_VIDEO_DEMO_CHECKPOINT = env_optional_str("RAW_VIDEO_DEMO_CHECKPOINT", None)
RAW_VIDEO_DEMO_SAMPLE_INDICES = (8, 9, 10, 11, 12, 13, 14, 15)
RAW_VIDEO_DEMO_TOPK = 5
RAW_VIDEO_DEMO_PRIOR_LAMBDA = 0.1
RAW_VIDEO_DEMO_ACTION_SCORE_WEIGHT = 2.0
RAW_VIDEO_DEMO_FRAME_COUNT = 6
RAW_VIDEO_DEMO_DIR = OUTPUT_DIR / "raw_video_demo"
RAW_VIDEO_SUFFIXES = (".mp4", ".avi", ".mkv", ".mov", ".webm", ".MP4", ".AVI", ".MKV", ".MOV", ".WEBM")
RAW_VIDEO_DEMO_CHECKPOINT = None

def find_discovered_file(name: str) -> Optional[Path]:
    name_lower = name.lower()
    for paths in DISCOVERED_FILES.values():
        for path in paths:
            if path.name.lower() == name_lower:
                return path
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    return None


def read_epic_annotation_rows(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.reader(handle)
        for raw in reader:
            if not raw or raw[0].strip().lower() in {"id", "uid"}:
                continue
            if len(raw) < 7:
                continue
            try:
                rows.append(
                    {
                        "uid": raw[0].strip(),
                        "video_id": raw[1].strip(),
                        "start_frame": int(raw[2]),
                        "stop_frame": int(raw[3]),
                        "verb": int(raw[4]),
                        "noun": int(raw[5]),
                        "action_id": int(raw[6]),
                    }
                )
            except ValueError:
                continue
    return rows


def load_action_name_maps(actions_path: Optional[Path]) -> Tuple[Dict[Tuple[int, int], str], Dict[int, str]]:
    by_pair: Dict[Tuple[int, int], str] = {}
    by_id: Dict[int, str] = {}
    if actions_path is None or not actions_path.exists():
        return by_pair, by_id
    with actions_path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            try:
                action_id = int(row["id"])
                verb = int(row["verb"])
                noun = int(row["noun"])
                action = str(row["action"])
            except (KeyError, TypeError, ValueError):
                continue
            by_pair[(verb, noun)] = action
            by_id[action_id] = action
    return by_pair, by_id


def find_raw_videos(video_ids: Sequence[str]) -> Dict[str, Path]:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    found: Dict[str, Path] = {}
    for video_id in video_ids:
        for root in roots:
            if video_id in found or not root.exists():
                continue
            for suffix in RAW_VIDEO_SUFFIXES:
                for candidate in root.rglob(f"{video_id}*{suffix}"):
                    if candidate.stem == video_id:
                        found[video_id] = candidate
                        break
                if video_id in found:
                    break
    return found


def find_joint_gru_checkpoint() -> Optional[Path]:
    if RAW_VIDEO_DEMO_CHECKPOINT:
        configured = Path(RAW_VIDEO_DEMO_CHECKPOINT)
        if configured.exists():
            return configured
        print(f"[raw-demo] configured checkpoint not found: {configured}")

    roots = [CHECKPOINT_DIR, OUTPUT_DIR / "runs", OUTPUT_DIR, Path("/kaggle/input"), Path.cwd()]
    patterns = [
        "**/joint_gru_seed7_best.pt",
        "**/joint_gru_seed123_best.pt",
        "**/joint_gru_seed42_best.pt",
        "**/joint_gru_best.pt",
    ]
    candidates: List[Path] = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(root.glob(pattern))
    if not candidates:
        return None
    unique_candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    return unique_candidates[0]


def infer_gru_hidden_dim_from_state(state_dict: Mapping[str, torch.Tensor]) -> int:
    for key, value in state_dict.items():
        normalized = key.removeprefix("module.")
        if normalized == "gru.weight_ih_l0":
            return int(value.shape[0] // 3)
    raise ValueError("Could not infer hidden_dim from checkpoint; missing gru.weight_ih_l0")


def load_demo_joint_gru(checkpoint_path: Path, device: torch.device) -> Optional[nn.Module]:
    try:
        loaded = torch.load(checkpoint_path, map_location=device)
        state_dict = loaded.get("state_dict", loaded) if isinstance(loaded, dict) else loaded
        if not isinstance(state_dict, Mapping):
            raise TypeError("checkpoint does not contain a state_dict mapping")
        state_dict = {
            key.removeprefix("module."): value
            for key, value in state_dict.items()
            if isinstance(value, torch.Tensor)
        }
        hidden_dim = infer_gru_hidden_dim_from_state(state_dict)
        model = JointGRUModel(
            FEATURE_DIM,
            NUM_VERBS,
            NUM_NOUNS,
            NUM_ACTIONS,
            hidden_dim=hidden_dim,
            dropout=0.0,
        )
        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        model.eval()
        print(f"[raw-demo] loaded checkpoint: {checkpoint_path}")
        print(f"[raw-demo] inferred JointGRU hidden_dim={hidden_dim}")
        return model
    except Exception as exc:
        print(f"[raw-demo] could not load checkpoint {checkpoint_path}: {exc}")
        return None


def topk_action_predictions(
    model: nn.Module,
    sample_index: int,
    topk: int,
    prior_lambda: float,
    action_score_weight: float,
    device: torch.device,
) -> List[Dict[str, Any]]:
    collate_fn = make_collate_fn(CFG.max_len, CFG.truncate_mode)
    batch = collate_fn([val_dataset[sample_index]])
    batch = move_batch_to_device(batch, device)
    with torch.no_grad(), autocast_context(device):
        verb_logits, noun_logits, action_logits = unpack_model_outputs(model(batch["features"], batch["mask"]))
        joint_scores = verb_logits.unsqueeze(2) + noun_logits.unsqueeze(1)
        if prior_lambda != 0.0:
            joint_scores = joint_scores + prior_lambda * LOG_PRIOR.to(device).unsqueeze(0)
        flat_scores = joint_scores.flatten(start_dim=1)
        if action_logits is not None and action_score_weight != 0.0:
            flat_indices = ACTION_FLAT_INDICES.to(device)
            bonus = torch.zeros_like(flat_scores)
            expanded_indices = flat_indices.unsqueeze(0).expand(action_logits.shape[0], -1)
            bonus.scatter_add_(1, expanded_indices, action_score_weight * action_logits.to(flat_scores.dtype))
            flat_scores = flat_scores + bonus

        values, indices = flat_scores.topk(min(topk, flat_scores.shape[1]), dim=1)

    predictions = []
    for rank, (score, flat_index) in enumerate(zip(values[0].detach().cpu(), indices[0].detach().cpu()), start=1):
        flat_int = int(flat_index)
        predictions.append(
            {
                "rank": rank,
                "score": float(score),
                "verb": flat_int // NUM_NOUNS,
                "noun": flat_int % NUM_NOUNS,
            }
        )
    return predictions


def save_video_contact_sheet(
    video_path: Path,
    start_frame: int,
    stop_frame: int,
    title: str,
    output_path: Path,
    frame_count: int,
) -> bool:
    if plt is None:
        print("[raw-demo] matplotlib unavailable; skipping contact sheet")
        return False
    try:
        import cv2
    except ImportError:
        print("[raw-demo] cv2 unavailable; skipping contact sheet")
        return False

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[raw-demo] could not open video: {video_path}")
        return False

    frame_ids = np.linspace(start_frame, stop_frame, num=max(1, frame_count), dtype=int)
    frames: List[Tuple[int, Any]] = []
    for frame_id in frame_ids:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(frame_id)))
        ok, frame = cap.read()
        if ok and frame is not None:
            frames.append((int(frame_id), cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()

    if not frames:
        print(f"[raw-demo] no frames extracted from {video_path}")
        return False

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, len(frames), figsize=(2.4 * len(frames), 2.4))
    if len(frames) == 1:
        axes = [axes]
    for axis, (frame_id, frame) in zip(axes, frames):
        axis.imshow(frame)
        axis.set_title(f"f={frame_id}", fontsize=8)
        axis.axis("off")
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)
    return True


def run_raw_video_demo() -> None:
    if not RUN_RAW_VIDEO_DEMO:
        print("[raw-demo] disabled. Set RUN_RAW_VIDEO_DEMO=True to generate qualitative assets.")
        return

    validation_path = find_discovered_file("validation.csv")
    actions_path = find_discovered_file("actions.csv")
    if validation_path is None:
        print("[raw-demo] validation.csv not found; skipping demo")
        return

    annotations = read_epic_annotation_rows(validation_path)
    if not annotations:
        print(f"[raw-demo] no annotation rows found in {validation_path}; skipping demo")
        return

    checkpoint_path = find_joint_gru_checkpoint()
    if checkpoint_path is None:
        print("[raw-demo] no JointGRU checkpoint found; skipping predictions")
        return
    model = load_demo_joint_gru(checkpoint_path, DEVICE)
    if model is None:
        return

    selected_indices = [
        int(index)
        for index in RAW_VIDEO_DEMO_SAMPLE_INDICES
        if 0 <= int(index) < min(len(val_dataset), len(annotations))
    ]
    if not selected_indices:
        print("[raw-demo] no valid sample indices selected")
        return

    action_by_pair, action_by_id = load_action_name_maps(actions_path)
    selected_video_ids = sorted({annotations[index]["video_id"] for index in selected_indices})
    video_paths = find_raw_videos(selected_video_ids)
    missing_videos = sorted(set(selected_video_ids) - set(video_paths))
    if missing_videos:
        print(f"[raw-demo] raw videos not found for: {missing_videos}")
        print("[raw-demo] predictions CSV will still be saved; contact sheets need raw video files.")

    demo_rows: List[Dict[str, Any]] = []
    demo_dir = Path(RAW_VIDEO_DEMO_DIR)
    if demo_dir.suffix in RAW_VIDEO_SUFFIXES:
        print(
            f"[raw-demo] RAW_VIDEO_DEMO_DIR points to a video file ({demo_dir}); "
            "using the default output directory instead."
        )
        demo_dir = OUTPUT_DIR / "raw_video_demo"
    demo_dir.mkdir(parents=True, exist_ok=True)

    for sample_index in selected_indices:
        annotation = annotations[sample_index]
        features, true_verb, true_noun = val_dataset[sample_index]
        predictions = topk_action_predictions(
            model,
            sample_index,
            RAW_VIDEO_DEMO_TOPK,
            RAW_VIDEO_DEMO_PRIOR_LAMBDA,
            RAW_VIDEO_DEMO_ACTION_SCORE_WEIGHT,
            DEVICE,
        )

        true_pair = (int(true_verb), int(true_noun))
        true_action_name = action_by_pair.get(
            true_pair,
            action_by_id.get(int(annotation["action_id"]), f"verb_{true_pair[0]}_noun_{true_pair[1]}"),
        )
        top1 = predictions[0]
        top1_name = action_by_pair.get((top1["verb"], top1["noun"]), f"verb_{top1['verb']}_noun_{top1['noun']}")

        for prediction in predictions:
            pred_pair = (int(prediction["verb"]), int(prediction["noun"]))
            demo_rows.append(
                {
                    "sample_index": sample_index,
                    "uid": annotation["uid"],
                    "video_id": annotation["video_id"],
                    "start_frame": annotation["start_frame"],
                    "stop_frame": annotation["stop_frame"],
                    "gt_verb": true_pair[0],
                    "gt_noun": true_pair[1],
                    "gt_action": true_action_name,
                    "rank": prediction["rank"],
                    "pred_verb": pred_pair[0],
                    "pred_noun": pred_pair[1],
                    "pred_action": action_by_pair.get(pred_pair, f"verb_{pred_pair[0]}_noun_{pred_pair[1]}"),
                    "score": prediction["score"],
                    "top1_correct": int((top1["verb"], top1["noun"]) == true_pair),
                    "top5_contains_gt": int(any((p["verb"], p["noun"]) == true_pair for p in predictions)),
                    "checkpoint": str(checkpoint_path),
                    "prior_lambda": RAW_VIDEO_DEMO_PRIOR_LAMBDA,
                    "action_score_weight": RAW_VIDEO_DEMO_ACTION_SCORE_WEIGHT,
                    "feature_shape": tuple(features.shape),
                }
            )

        video_path = video_paths.get(annotation["video_id"])
        if video_path is not None:
            title = (
                f"{annotation['video_id']} sample {sample_index}: "
                f"GT={true_action_name}, top1={top1_name}"
            )
            image_path = demo_dir / f"sample_{sample_index:05d}_{annotation['video_id']}.png"
            if save_video_contact_sheet(
                video_path,
                annotation["start_frame"],
                annotation["stop_frame"],
                title,
                image_path,
                RAW_VIDEO_DEMO_FRAME_COUNT,
            ):
                print(f"[raw-demo] saved contact sheet: {image_path}")

    predictions_path = demo_dir / "demo_predictions.csv"
    save_results_table(demo_rows, predictions_path)
    print(f"[raw-demo] saved predictions: {predictions_path}")


run_raw_video_demo()


[raw-demo] loaded checkpoint: /kaggle/working/runs/joint_gru768_actionscore_highgamma_real_joint_gru_h768_l14_b256_e60_lr2em04_wd0p01_do0p35_seeds42-123-7_alw0p3_p20_asws1-1p25-1p5-2_mep20_trfull_vafull_d1024_v119_n321/checkpoints/joint_gru_seed7_best.pt
[raw-demo] inferred JointGRU hidden_dim=768
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00000_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00001_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00002_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00003_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00004_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00005_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00006_P01_01.png
[raw-demo] saved contact sheet: /kaggle/working/raw_video_demo/sample_00007_P01_01.png
[raw-